# HAM10000 Skin Lesion Classification
### Complete Notebook — High Accuracy | Google Drive | Auto-Resume
- Disk loading — starts immediately, no RAM issues
- 2-phase training (4+6 epochs) per model
- Mixed precision — no OOM errors
- Auto-saves to Drive after every model
- Resumes automatically if Colab disconnects


## Step 1 — Install

In [6]:
!pip install kagglehub timm xgboost thop -q
print("✅ Done")

✅ Done


## Step 2 — Mount Google Drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/HAM10000_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Drive mounted → {DRIVE_DIR}")

Mounted at /content/drive
✅ Drive mounted → /content/drive/MyDrive/HAM10000_Results


## Step 3 — All Imports & Config

In [8]:
import os, gc, time, copy, json, shutil, warnings
import numpy as np
import pandas as pd
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing  import label_binarize
from sklearn.linear_model   import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from xgboost                 import XGBClassifier

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except: THOP_AVAILABLE = False

# ── Paths ──────────────────────────────────────
SAVE_DIR  = './results'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Memory fix ─────────────────────────────────
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")


Device : cuda
GPU    : Tesla T4
Memory : 14.6 GB


## Step 4 — Download HAM10000

In [9]:
import kagglehub
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Downloaded to:", path)

# Find CSV
csv_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if 'metadata' in f.lower() and f.endswith('.csv'):
            csv_path = os.path.join(root, f); break
    if csv_path: break
print("CSV:", csv_path)

# Map all images
img_map = {}
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith('.jpg'):
            img_map[os.path.splitext(f)[0]] = os.path.join(root, f)
print(f"Images: {len(img_map)}")

# Load metadata
df          = pd.read_csv(csv_path)
df          = df.drop_duplicates(subset='lesion_id', keep='first')
df          = df[df['image_id'].isin(img_map)].reset_index(drop=True)
df['path']  = df['image_id'].map(img_map)
CLASSES     = sorted(df['dx'].unique())
NUM_CLASSES = len(CLASSES)
cls2idx     = {c:i for i,c in enumerate(CLASSES)}
df['label'] = df['dx'].map(cls2idx)
print(f"Classes ({NUM_CLASSES}): {CLASSES}")
print(df['dx'].value_counts())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Downloaded to: /kaggle/input/skin-cancer-mnist-ham10000
CSV: /kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv
Images: 10015
Classes (7): ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64


## Step 5 — Split Data

In [10]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=42)
train_df, val_df  = train_test_split(
    train_df, test_size=0.125, stratify=train_df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train:{len(train_df)}  Val:{len(val_df)}  Test:{len(test_df)}")

Train:5229  Val:747  Test:1494


## Step 6 — Dataset & Transforms

In [11]:
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class DiskDataset(Dataset):
    """Loads images from disk — no RAM caching, starts instantly."""
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label'])

def make_loaders(bs=32):
    labels       = train_df['label'].values
    class_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights      = 1.0 / class_counts[labels]
    sampler      = WeightedRandomSampler(weights, len(weights))
    tr = DataLoader(DiskDataset(train_df, train_tf), batch_size=bs,
                    sampler=sampler, num_workers=4, pin_memory=True)
    vl = DataLoader(DiskDataset(val_df,   val_tf),  batch_size=bs,
                    shuffle=False,  num_workers=4, pin_memory=True)
    te = DataLoader(DiskDataset(test_df,  val_tf),  batch_size=bs,
                    shuffle=False,  num_workers=4, pin_memory=True)
    return tr, vl, te

print("✅ Dataset ready")

✅ Dataset ready


## Step 7 — Model Factory

In [12]:
# Batch size per model (smaller for large models to avoid OOM)
BATCH_SIZE_MAP = {
    'AlexNet':32, 'VGG16':16, 'VGG19':16,
    'ResNet18':32, 'ResNet50':32, 'ResNet101':8,
    'DenseNet121':16, 'EfficientNet-B0':32,
}

def get_model(name):
    n = name.lower().replace('-','').replace('_','')

    if n == 'alexnet':
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'vgg16':
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'vgg19':
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.classifier.parameters(): p.requires_grad = True
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif n == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer3.parameters(): p.requires_grad = True
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer3.parameters(): p.requires_grad = True
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'resnet101':
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.layer4.parameters(): p.requires_grad = True
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)

    elif n == 'densenet121':
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        for p in m.features.denseblock4.parameters(): p.requires_grad = True
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)

    elif n == 'efficientnetb0':
        m = timm.create_model('efficientnet_b0', pretrained=True,
                               num_classes=NUM_CLASSES)
        for p in m.parameters(): p.requires_grad = False
        for p in m.blocks[-3].parameters(): p.requires_grad = True
        for p in m.blocks[-2].parameters(): p.requires_grad = True
        for p in m.blocks[-1].parameters(): p.requires_grad = True
        for p in m.classifier.parameters(): p.requires_grad = True
    else:
        raise ValueError(f"Unknown: {name}")

    tr  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    print(f"  {name:<18} trainable:{tr:>10,}/{tot:>10,} ({100*tr/tot:.1f}%)")
    return m

MODEL_NAMES = ['AlexNet','VGG16','VGG19',
               'ResNet18','ResNet50','ResNet101',
               'DenseNet121','EfficientNet-B0']
print("Model summary:")
for mn in MODEL_NAMES:
    m = get_model(mn); del m
print("\n✅ Model factory ready")

Model summary:
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 185MB/s]


  AlexNet            trainable:54,562,823/57,032,519 (95.7%)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 177MB/s]


  VGG16              trainable:119,574,535/134,289,223 (89.0%)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 153MB/s]


  DenseNet121        trainable: 2,165,255/ 6,961,031 (31.1%)


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  EfficientNet-B0    trainable: 3,295,695/ 4,016,515 (82.1%)

✅ Model factory ready


## Step 8 — Training & Evaluation Helpers

In [13]:
def train_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    tl = cr = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        tl  += loss.item() * imgs.size(0)
        cr  += (out.argmax(1) == labels).sum().item()
        tot += imgs.size(0)
    return tl/tot, cr/tot

def evaluate(model, loader):
    model.eval()
    ap, al, ab = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            out = model(imgs.to(DEVICE))
            ab.extend(torch.softmax(out,1).cpu().numpy())
            ap.extend(out.argmax(1).cpu().numpy())
            al.extend(labels.numpy())
    yt = np.array(al); yp = np.array(ap); yb = np.array(ab)
    acc  = accuracy_score(yt,yp)*100
    prec = precision_score(yt,yp,average='weighted',zero_division=0)*100
    rec  = recall_score(yt,yp,   average='weighted',zero_division=0)*100
    f1   = f1_score(yt,yp,        average='weighted',zero_division=0)*100
    try:
        ybin = label_binarize(yt, classes=list(range(NUM_CLASSES)))
        auc  = roc_auc_score(ybin,yb,multi_class='ovr',average='weighted')*100
    except: auc = 0.0
    return round(acc,2),round(prec,2),round(rec,2),round(f1,2),round(auc,2)

def compute_efficiency(model):
    dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
    model.eval()
    pm = sum(p.numel() for p in model.parameters())/1e6
    sm = sum(p.numel()*p.element_size() for p in model.parameters())/(1024**2)
    fg = 'N/A'
    if THOP_AVAILABLE:
        try:
            fl,_ = thop_profile(model,inputs=(dummy,),verbose=False)
            fg   = round(fl/1e9,2)
        except: pass
    with torch.no_grad():
        for _ in range(5): model(dummy)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(50): model(dummy)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        im = (time.perf_counter()-t0)/50*1000
    return round(pm,2),round(sm,2),fg,round(im,2)

def save_to_drive(filename):
    shutil.copy2(os.path.join(SAVE_DIR,filename),
                 os.path.join(DRIVE_DIR,filename))
    print(f"  ☁️  {filename}")

print("✅ Helpers ready")

✅ Helpers ready


## Step 9 — Train All 8 Models → Table 1 + Table 3
**2-Phase Training per model:**
- Phase 1 (4 epochs, LR=1e-3): Warm up classifier + last blocks
- Phase 2 (6 epochs, LR=1e-5): Fine-tune all layers
- Mixed precision → no OOM
- Auto-resumes from Drive if Colab disconnects


In [14]:
import os
import json
import shutil
import copy
import gc
import time
import pandas as pd
import torch.nn as nn
import torch.optim as optim

DRIVE_DIR = '/content/drive/MyDrive/HAM10000_Results'
SAVE_DIR  = './results'

EPOCHS_P1 = 4
EPOCHS_P2 = 6
LR_P1     = 1e-3
LR_P2     = 1e-5

# Batch size per model (smaller for large models to avoid OOM)
BATCH_SIZE_MAP = {
    'AlexNet':32, 'VGG16':16, 'VGG19':16,
    'ResNet18':32, 'ResNet50':32, 'ResNet101':8,
    'DenseNet121':16, 'EfficientNet-B0':32,
}

MODEL_NAMES = ['AlexNet','VGG16','VGG19',
               'ResNet18','ResNet50','ResNet101',
               'DenseNet121','EfficientNet-B0']

# ── Load existing results from Drive ───────────
results_json_drive = os.path.join(DRIVE_DIR,'all_results.json')
results_json_local = os.path.join(SAVE_DIR, 'all_results.json')

if os.path.exists(results_json_drive):
    shutil.copy2(results_json_drive, results_json_local)
    with open(results_json_local) as f: saved = json.load(f)
    table1 = saved.get('table1',{})
    table3 = saved.get('table3',{})
    print(f"✅ Resumed from Drive. Done: {list(table1.keys())}")
else:
    table1, table3 = {}, {}
    print("🆕 Starting fresh")

best_backbone = max(table1,key=lambda m:table1[m]['Accuracy (%)'])                 if table1 else None
best_acc = table1[best_backbone]['Accuracy (%)'] if best_backbone else 0.0

for mname in MODEL_NAMES:

    ckpt_name  = mname.replace('-','_')+'_best.pth'
    ckpt_local = os.path.join(SAVE_DIR, ckpt_name)
    ckpt_drive = os.path.join(DRIVE_DIR,ckpt_name)

    # Restore checkpoint from Drive if session reset
    if os.path.exists(ckpt_drive) and not os.path.exists(ckpt_local):
        shutil.copy2(ckpt_drive, ckpt_local)
        print(f"  📥 Restored {ckpt_name}")

    # Skip already trained models
    if mname in table1 and os.path.exists(ckpt_local):
        print(f"  ⏭️  {mname:<18} Acc:{table1[mname]['Accuracy (%)']}%")
        if table1[mname]['Accuracy (%)'] > best_acc:
            best_acc = table1[mname]['Accuracy (%)']
            best_backbone = mname
        continue

    # Per-model batch size to avoid OOM
    bs = BATCH_SIZE_MAP.get(mname, 16)
    print(f"\n{'='*55}\n  Training: {mname}  (batch={bs})\n{'='*55}")

    # Fresh loaders per model
    torch.cuda.empty_cache(); gc.collect()
    tr_l, vl_l, te_l = make_loaders(bs)
    model     = get_model(mname).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type=='cuda' else None
    best_val, best_w = 0.0, None

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  PHASE 1 — Warm up
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n  📌 Phase 1 ({EPOCHS_P1} epochs | LR={LR_P1})")
    opt1  = optim.Adam(
        filter(lambda p:p.requires_grad, model.parameters()),
        lr=LR_P1, weight_decay=1e-4)
    sch1  = optim.lr_scheduler.CosineAnnealingLR(
        opt1, T_max=EPOCHS_P1, eta_min=1e-5)

    for ep in range(1, EPOCHS_P1+1):
        t0 = time.time()
        tl,ta = train_epoch(model,tr_l,criterion,opt1,scaler)
        va,*_ = evaluate(model,vl_l)
        sch1.step()
        print(f"  Ep{ep:02d}/{EPOCHS_P1} Loss:{tl:.4f} "
              f"Train:{ta*100:.1f}% Val:{va:.1f}% "
              f"LR:{opt1.param_groups[0]['lr']:.1e} {time.time()-t0:.0f}s")
        if va > best_val: best_val=va; best_w=copy.deepcopy(model.state_dict())

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  PHASE 2 — Fine-tune all layers
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print(f"\n  🔓 Phase 2 ({EPOCHS_P2} epochs | LR={LR_P2})")
    for p in model.parameters(): p.requires_grad = True
    torch.cuda.empty_cache()

    opt2  = optim.Adam(model.parameters(), lr=LR_P2, weight_decay=1e-4)
    sch2  = optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=EPOCHS_P2, eta_min=1e-7)

    for ep in range(1, EPOCHS_P2+1):
        t0 = time.time()
        tl,ta = train_epoch(model,tr_l,criterion,opt2,scaler)
        va,*_ = evaluate(model,vl_l)
        sch2.step()
        print(f"  Ep{ep:02d}/{EPOCHS_P2} Loss:{tl:.4f} "
              f"Train:{ta*100:.1f}% Val:{va:.1f}% "
              f"LR:{opt2.param_groups[0]['lr']:.1e} {time.time()-t0:.0f}s")
        if va > best_val: best_val=va; best_w=copy.deepcopy(model.state_dict())

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    #  TEST
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    model.load_state_dict(best_w)
    acc,prec,rec,f1,auc = evaluate(model, te_l)
    print(f"\n  ✔ TEST Acc:{acc} Prec:{prec} Rec:{rec} F1:{f1} AUC:{auc}")

    # Save checkpoint + results
    torch.save(model.state_dict(), ckpt_local)
    save_to_drive(ckpt_name)

    p,s,fl,im = compute_efficiency(model)
    table1[mname] = {'Accuracy (%)':acc,'Precision (%)':prec,
                     'Recall (%)':rec,  'F1-Score (%)':f1,'AUC (%)':auc}
    table3[mname] = {'Parameters (M)':p,'Model Size (MB)':s,
                     'FLOPs (G)':fl,    'Inference Time (ms)':im,'Accuracy (%)':acc}
    if acc > best_acc: best_acc=acc; best_backbone=mname

    with open(results_json_local,'w') as f:
        json.dump({'table1':table1,'table3':table3,
                   'best_backbone':best_backbone},f,indent=2)
    pd.DataFrame(table1).T.to_csv(os.path.join(SAVE_DIR,'table1.csv'))
    pd.DataFrame(table3).T.to_csv(os.path.join(SAVE_DIR,'table3.csv'))
    save_to_drive('all_results.json')
    save_to_drive('table1.csv')
    save_to_drive('table3.csv')
    print(f"  ✅ {mname} saved to Drive! Done:{list(table1.keys())}")

    del model, tr_l, vl_l, te_l
    torch.cuda.empty_cache(); gc.collect()

print(f"\n🎉 Table 1 + 3 complete! Best: {best_backbone} ({best_acc}%)")

✅ Resumed from Drive. Done: ['AlexNet', 'VGG16', 'VGG19']
  📥 Restored AlexNet_best.pth
  ⏭️  AlexNet            Acc:66.2%
  📥 Restored VGG16_best.pth
  ⏭️  VGG16              Acc:66.2%
  📥 Restored VGG19_best.pth
  ⏭️  VGG19              Acc:66.27%

  Training: ResNet18  (batch=32)
  ResNet18           trainable:10,497,031/11,180,103 (93.9%)

  📌 Phase 1 (4 epochs | LR=0.001)
  Ep01/4 Loss:1.3018 Train:60.0% Val:72.3% LR:8.6e-04 75s
  Ep02/4 Loss:1.0545 Train:71.5% Val:70.2% LR:5.1e-04 72s
  Ep03/4 Loss:0.9417 Train:77.3% Val:69.2% LR:1.5e-04 73s
  Ep04/4 Loss:0.8311 Train:82.5% Val:70.4% LR:1.0e-05 72s

  🔓 Phase 2 (6 epochs | LR=1e-05)
  Ep01/6 Loss:0.7891 Train:85.1% Val:75.0% LR:9.3e-06 73s
  Ep02/6 Loss:0.7845 Train:84.8% Val:74.6% LR:7.5e-06 72s
  Ep03/6 Loss:0.7542 Train:86.3% Val:74.7% LR:5.0e-06 72s
  Ep04/6 Loss:0.7660 Train:85.6% Val:74.2% LR:2.6e-06 73s
  Ep05/6 Loss:0.7629 Train:86.0% Val:74.8% LR:7.6e-07 73s
  Ep06/6 Loss:0.7722 Train:85.4% Val:76.0% LR:1.0e-07 73s

  ✔ 

## Step 10 — Deep Features + 7 Classifiers → Table 2

In [16]:
print(f"Extracting features from: {best_backbone}")

ckpt_local = os.path.join(SAVE_DIR, best_backbone.replace('-','_')+'_best.pth')
ckpt_drive = os.path.join(DRIVE_DIR,best_backbone.replace('-','_')+'_best.pth')
if not os.path.exists(ckpt_local) and os.path.exists(ckpt_drive):
    shutil.copy2(ckpt_drive, ckpt_local)

model = get_model(best_backbone)
model.load_state_dict(torch.load(ckpt_local, map_location=DEVICE))
model = model.to(DEVICE).eval()
for p in model.parameters(): p.requires_grad = False

# Hook on penultimate layer
captured = []
def hook(m,i,o): captured.append(o.detach().cpu())

n = best_backbone.lower().replace('-','').replace('_','')
if 'alexnet' in n or 'vgg' in n:
    handle = model.classifier[-2].register_forward_hook(hook)
elif 'resnet' in n:
    handle = model.avgpool.register_forward_hook(hook)
elif 'densenet' in n:
    handle = model.features.register_forward_hook(hook)
else:
    handle = model.global_pool.register_forward_hook(hook)

bs = BATCH_SIZE_MAP.get(best_backbone, 32)
tr_l, _, te_l = make_loaders(bs)

def collect(loader):
    F,L=[],[]
    with torch.no_grad():
        for imgs,labels in loader:
            captured.clear()
            model(imgs.to(DEVICE))
            f = captured[0].view(captured[0].size(0),-1)
            F.append(f.numpy()); L.extend(labels.numpy())
    return np.vstack(F), np.array(L)

X_tr,y_tr = collect(tr_l)
X_te,y_te = collect(te_l)
handle.remove(); del model
torch.cuda.empty_cache(); gc.collect()
print(f"Features — Train:{X_tr.shape}  Test:{X_te.shape}")

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000,n_jobs=-1),
    'Decision Tree':       DecisionTreeClassifier(max_depth=15,random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5,n_jobs=-1),
    'Linear SVM':          SVC(kernel='linear',probability=True,max_iter=2000),
    'RBF-SVM':             SVC(kernel='rbf',probability=True,C=10,gamma='scale'),
    'XGBoost':             XGBClassifier(n_estimators=200,learning_rate=0.1,
                                          eval_metric='mlogloss',n_jobs=-1,random_state=42),
}
table2 = {}
for cn,clf in classifiers.items():
    print(f"  {cn}...",end=' ',flush=True)
    t0=time.time(); clf.fit(X_tr,y_tr); print(f"{time.time()-t0:.1f}s")
    preds=clf.predict(X_te)
    acc =accuracy_score(y_te,preds)*100
    prec=precision_score(y_te,preds,average='weighted',zero_division=0)*100
    rec =recall_score(y_te,preds,   average='weighted',zero_division=0)*100
    f1  =f1_score(y_te,preds,        average='weighted',zero_division=0)*100
    try:
        prob =clf.predict_proba(X_te) if hasattr(clf,'predict_proba')               else np.eye(NUM_CLASSES)[preds]
        ybin =label_binarize(y_te,classes=list(range(NUM_CLASSES)))
        auc  =roc_auc_score(ybin,prob,multi_class='ovr',average='weighted')*100
    except: auc=0.0
    table2[cn]={'Feature Extractor':best_backbone,'Classifier':cn,
                'Accuracy (%)':round(acc,2),'Precision (%)':round(prec,2),
                'Recall (%)':round(rec,2),'F1-Score (%)':round(f1,2),
                'AUC (%)':round(auc,2)}
    print(f"    Acc:{round(acc,2)} Prec:{round(prec,2)} "
          f"Rec:{round(rec,2)} F1:{round(f1,2)} AUC:{round(auc,2)}")

pd.DataFrame(table2).T.to_csv(os.path.join(SAVE_DIR,'table2.csv'))
save_to_drive('table2.csv')
print("\n✅ Table 2 done!")

Extracting features from: ResNet101
  ResNet101          trainable:14,979,079/42,514,503 (35.2%)
Features — Train:(5229, 2048)  Test:(1494, 2048)
  Logistic Regression... 6.8s
    Acc:78.65 Prec:82.85 Rec:78.65 F1:80.21 AUC:94.13
  Decision Tree... 15.1s
    Acc:67.74 Prec:78.72 Rec:67.74 F1:71.49 AUC:77.35
  Random Forest... 33.6s
    Acc:76.31 Prec:82.57 Rec:76.31 F1:78.45 AUC:94.25
  K-Nearest Neighbors... 0.0s
    Acc:74.1 Prec:81.27 Rec:74.1 F1:76.69 AUC:90.48
  Linear SVM... 22.1s
    Acc:77.78 Prec:81.69 Rec:77.78 F1:79.32 AUC:93.71
  RBF-SVM... 43.2s
    Acc:79.18 Prec:82.75 Rec:79.18 F1:80.55 AUC:94.09
  XGBoost... 443.0s
    Acc:77.04 Prec:82.89 Rec:77.04 F1:79.14 AUC:94.39
  ☁️  table2.csv

✅ Table 2 done!


## Step 11 — Print All Tables & Save Everything

In [17]:
S = "─"*76

print(f"{'═'*76}")
print("  TABLE 1 — Transfer Learning Model Comparison")
print(f"{'═'*76}")
print(f"{'Model':<18}{'Acc%':>9}{'Prec%':>9}{'Rec%':>9}{'F1%':>9}{'AUC%':>9}")
print(S)
for m,r in table1.items():
    print(f"{m:<18}{r['Accuracy (%)']:>9}{r['Precision (%)']:>9}"
          f"{r['Recall (%)']:>9}{r['F1-Score (%)']:>9}{r['AUC (%)']:>9}")

print(f"\n{'═'*76}")
print("  TABLE 2 — Deep Features + Classical Classifiers")
print(f"{'═'*76}")
print(f"{'Classifier':<25}{'Acc%':>9}{'Prec%':>9}{'Rec%':>9}{'F1%':>9}{'AUC%':>9}")
print(S)
for c,r in table2.items():
    print(f"{c:<25}{r['Accuracy (%)']:>9}{r['Precision (%)']:>9}"
          f"{r['Recall (%)']:>9}{r['F1-Score (%)']:>9}{r['AUC (%)']:>9}")

print(f"\n{'═'*76}")
print("  TABLE 3 — Computational Efficiency")
print(f"{'═'*76}")
print(f"{'Model':<18}{'Params(M)':>11}{'Size(MB)':>10}"
      f"{'FLOPs(G)':>10}{'Infer(ms)':>11}{'Acc%':>9}")
print(S)
for m,r in table3.items():
    print(f"{m:<18}{r['Parameters (M)']:>11}{r['Model Size (MB)']:>10}"
          f"{str(r['FLOPs (G)']):>10}{r['Inference Time (ms)']:>11}"
          f"{r['Accuracy (%)']:>9}")

# Final save all
with open(results_json_local,'w') as f:
    json.dump({'table1':table1,'table2':table2,
               'table3':table3,'best_backbone':best_backbone},f,indent=2)
pd.DataFrame(table1).T.to_csv(os.path.join(SAVE_DIR,'table1.csv'))
pd.DataFrame(table2).T.to_csv(os.path.join(SAVE_DIR,'table2.csv'))
pd.DataFrame(table3).T.to_csv(os.path.join(SAVE_DIR,'table3.csv'))
for fname in ['all_results.json','table1.csv','table2.csv','table3.csv']:
    save_to_drive(fname)

print(f"\n✅ ALL DONE!")
print(f"📁 Google Drive: {DRIVE_DIR}")
print("\nDrive contents:")
for f in sorted(os.listdir(DRIVE_DIR)):
    sz = os.path.getsize(os.path.join(DRIVE_DIR,f))/(1024*1024)
    print(f"  {f:<45} {sz:.1f} MB")
print("\n🎉 Copy numbers into your Task_01.docx tables!")

════════════════════════════════════════════════════════════════════════════
  TABLE 1 — Transfer Learning Model Comparison
════════════════════════════════════════════════════════════════════════════
Model                  Acc%    Prec%     Rec%      F1%     AUC%
────────────────────────────────────────────────────────────────────────────
AlexNet                66.2    81.29     66.2    70.73     92.0
VGG16                  66.2    79.49     66.2    70.46    91.18
VGG19                 66.27    81.29    66.27    70.87    91.34
ResNet18              75.37    83.64    75.37    78.14    94.84
ResNet50              79.12    84.25    79.12    80.93    95.34
ResNet101             79.18    83.36    79.18    80.81    94.61
DenseNet121           77.24    83.05    77.24    79.28    94.53
EfficientNet-B0        75.5    82.43     75.5    77.98    93.71

════════════════════════════════════════════════════════════════════════════
  TABLE 2 — Deep Features + Classical Classifiers
══════════════════